# BERDL Lakehouse — Ingest: `nmdc_nmdc_linkml_store`

Two-phase ingest: **(1) upload Parquet files to MinIO bronze**, then **(2) write Delta
tables to silver**. Progress is saved to MinIO so interrupted ingests can resume.

On JupyterHub, Spark and MinIO clients come from `berdl_notebook_utils` directly. The
ingest helpers (`upload_files`, `run_ingest`, `verify_ingest`) come from BERIL's
`scripts/ingest_lib.py`.

## Configuration

In [ ]:
from pathlib import Path

DATA_DIR        = Path("~/nmdc-lakehouse/lakehouse").expanduser()
TENANT          = "nmdc"
DATASET         = "nmdc_linkml_store"
BUCKET          = "cdm-lake"
MODE            = "overwrite"
CHUNK_TARGET_GB = 20
CHUNKED_INGEST  = True

NAMESPACE     = f"{TENANT}_{DATASET}"
BRONZE_PREFIX = f"tenant-general-warehouse/{TENANT}/datasets/{DATASET}"
SILVER_BASE   = f"s3a://{BUCKET}/tenant-sql-warehouse/{TENANT}/{NAMESPACE}.db"
CONFIG_KEY    = f"{BRONZE_PREFIX}/config/{DATASET}.json"
PROGRESS_KEY  = f"{BRONZE_PREFIX}/_ingest_progress.jsonl"

print(f"Data dir  : {DATA_DIR}")
print(f"Mode      : {MODE}  |  Chunk threshold: {CHUNK_TARGET_GB} GB  |  Chunked: {CHUNKED_INGEST}")
print(f"Namespace : {NAMESPACE}")
print(f"Bronze    : s3a://{BUCKET}/{BRONZE_PREFIX}/")
print(f"Silver    : {SILVER_BASE}")


## Setup — Spark and MinIO

Use `berdl_notebook_utils.get_spark_session()` and `get_minio_client()` (real, on-cluster) instead of `ingest_lib.initialize()` (which is for off-cluster bootstrap with SSH tunnels).

Then add `~/BERIL-research-observatory/scripts` to `sys.path` so `ingest_lib`'s ingest helpers are importable.

In [ ]:
import sys
from pathlib import Path

from berdl_notebook_utils import get_spark_session, get_minio_client

spark        = get_spark_session()
minio_client = get_minio_client()

_scripts_dir = Path("~/BERIL-research-observatory/scripts").expanduser()
if str(_scripts_dir) not in sys.path:
    sys.path.insert(0, str(_scripts_dir))

from ingest_lib import (
    detect_source_files,
    build_table_stats,
    upload_files,
    run_ingest,
    verify_ingest,
)

# ingest_lib's import-time stub setup nulls out these submodule attrs for off-cluster
# use. We have real clients on JupyterHub — re-bind so downstream auto-init still works.
sys.modules["berdl_notebook_utils.setup_spark_session"].get_spark_session = lambda **kw: spark
sys.modules["berdl_notebook_utils.clients"].get_minio_client = lambda **kw: minio_client

print("Spark and MinIO clients ready.")


## Detect — Source files

In [ ]:
SOURCE_MODE, SOURCE_DB, SQL_SCHEMA, data_files, FILE_EXT, DELIMITER = \
    detect_source_files(DATA_DIR)


## Schema

Parquet files carry their own schema — no parsing needed.

In [ ]:
SCHEMAS, SCHEMA_DEFS = {}, {}
print("Parquet mode — schema is embedded in the files, no parsing needed.")


## Plan — Per-table line counts and chunk plan

All Parquet files in this dataset are well below `CHUNK_TARGET_GB`, so each table is
ingested as a single batch.

In [ ]:
TABLE_STATS = build_table_stats(
    data_files, SCHEMAS, CHUNK_TARGET_GB, CHUNKED_INGEST, DELIMITER,
)


## Ingest — Upload and write Delta tables

In [ ]:
upload_files(
    minio_client=minio_client,
    bucket=BUCKET,
    table_stats=TABLE_STATS,
    bronze_prefix=BRONZE_PREFIX,
    file_ext=FILE_EXT,
)

spark = run_ingest(
    spark=spark,
    minio_client=minio_client,
    table_stats=TABLE_STATS,
    schemas=SCHEMAS,
    schema_defs=SCHEMA_DEFS,
    namespace=NAMESPACE,
    tenant=TENANT,
    dataset=DATASET,
    bucket=BUCKET,
    bronze_prefix=BRONZE_PREFIX,
    silver_base=SILVER_BASE,
    mode=MODE,
    file_ext=FILE_EXT,
    delimiter=DELIMITER,
    progress_key=PROGRESS_KEY,
    config_key=CONFIG_KEY,
)


## Verify — Row counts

In [ ]:
verify_ingest(
    spark=spark,
    namespace=NAMESPACE,
    table_stats=TABLE_STATS,
    minio_client=minio_client,
    bucket=BUCKET,
    progress_key=PROGRESS_KEY,
    silver_base=SILVER_BASE,
)
